In [36]:
import numpy as np
from case_studies import *

def backtrack(f, df, x, p, alpha_init=1.0, rho=0.5, c1=1e-4):
    fx = f(x)
    grad_x = df(x)
    alpha = alpha_init
    while f(x + alpha * p) > fx + c1 * alpha * np.dot(grad_x, p):
        alpha *= rho
    return alpha

def solve_linear_system(B, A, grad_f):
    m, n = A.shape
    zero_block = np.zeros((m, m))
    lhs = np.block([[B, A.T], [A, zero_block]])
    rhs = np.concatenate([-grad_f, np.zeros(m)])
    solution = np.linalg.solve(lhs, rhs)
    return solution[:n]  # Extract p_k

# Newton's Method function (updated version)
def newtons_method(f, df, Hf, x0, max_iter=10000, epsilon=1e-6):
    x = x0.copy()
    xs = [x]
    for _ in range(max_iter):
        if np.linalg.norm(df(x)) < epsilon: break
        H = Hf(x)
        if np.linalg.eigvals(H).min() > 0:  # Check if Hessian is positive definite
            p = -np.linalg.solve(H, df(x))
        else:
            eigvals, eigvecs = np.linalg.eigh(H)
            new_H = sum((1 / abs(l)) * np.outer(v, v) for l, v in zip(eigvals, eigvecs.T))
            p = -new_H @ df(x)
        alpha = backtrack(f, df, x, p, 1.0)
        x = x + alpha * p
        xs.append(x)
    return np.array(xs)

def constrained_newton(f, df, Hf, x0, A, c1=1e-4, rho=0.5, epsilon=1e-6, max_iter=10000):
    x_k = x0
    for _ in range(max_iter):
        if np.linalg.norm(df(x_k)) < epsilon: break
        grad_f = df(x_k)
        hessian_f = Hf(x_k)
        if np.linalg.eigvals(hessian_f).min() > 0:  # Check if Hessian is positive definite
            B = hessian_f
        else:
            eigvals, eigvecs = np.linalg.eigh(hessian_f)
            B = sum(abs(l) * np.outer(v, v) for l, v in zip(eigvals, eigvecs.T))
        p_k = solve_linear_system(B, A, grad_f)
        alpha_k = backtrack(f, df, x_k, p_k, 1, c1, rho)
        x_k = x_k + alpha_k * p_k
    return x_k

def constrained_steepest_descent(f, df, x0, A, c1=1e-4, rho=0.5, epsilon=1e-6, max_iter=10000):
    x_k = x0
    beta_k = 1
    M = np.eye(len(x0)) - A.T @ np.linalg.inv(A @ A.T) @ A
    for _ in range(max_iter):
        if np.linalg.norm(df(x_k)) < epsilon: break
        grad_f = df(x_k)
        p_k = -M @ grad_f
        alpha_k = backtrack(f, df, x_k, p_k, beta_k, c1, rho)
        x_k = x_k + alpha_k * p_k
        beta_k = alpha_k / rho
    return x_k

In [ ]:
def testPrint():
    # Dimensions for each function
    n1, n2, n3, n4, n5 = 5, 2, 5, 5, 5
    
    # Constraint matrices
    # Change based on which constraints we want to test
    A1 = np.random.randn(2, n1)  # 2 constraints for f1
    A2 = np.random.randn(1, n2)  # 1 constraint for f2
    A3 = np.random.randn(3, n3)  # 3 constraints for f3
    A4 = np.random.randn(1, n4)  # 1 constraint for f4
    A5 = np.random.randn(2, n5)  # 2 constraints for f5
    
    # Random b vectors
    b1 = np.random.randn(2)
    b2 = np.random.randn(1)
    b3 = np.random.randn(3)
    b4 = np.random.randn(1)
    b5 = np.random.randn(2)
    
    # Generate feasible initial points that satisfy Ax + b = 0
    x0_f1 = np.linalg.lstsq(A1, -b1, rcond=None)[0]
    x0_f2 = np.linalg.lstsq(A2, -b2, rcond=None)[0]
    x0_f3 = np.linalg.lstsq(A3, -b3, rcond=None)[0]
    x0_f4 = np.linalg.lstsq(A4, -b4, rcond=None)[0]
    x0_f5 = np.linalg.lstsq(A5, -b5, rcond=None)[0]

    # Running Steepest Descent
    print("\nNewton's Method Results:")
    print("f1:", newtons_method(f1, df1, Hf1, x0_f1)[0])
    print("f2:", newtons_method(f2, df2, Hf2, x0_f2)[0])
    print("f3:", newtons_method(f3, df3, Hf3, x0_f3)[0])
    print("f4:", newtons_method(f4, df4, Hf4, x0_f4)[0])
    print("f5:", newtons_method(f5, df5, Hf5, x0_f5)[0])

    # Running Constrained Newton's Method
    print("Constrained Newton's Method Results:")
    print("f1:", constrained_newton(f1, df1, Hf1, x0_f1, A1))
    print("f2:", constrained_newton(f2, df2, Hf2, x0_f2, A2))
    print("f3:", constrained_newton(f3, df3, Hf3, x0_f3, A3))
    print("f4:", constrained_newton(f4, df4, Hf4, x0_f4, A4))
    print("f5:", constrained_newton(f5, df5, Hf5, x0_f5, A5))

    # Running Constrained Steepest Descent
    print("Constrained Steepest Descent Results:")
    print("f1:", constrained_steepest_descent(f1, df1, x0_f1, A1))
    print("f2:", constrained_steepest_descent(f2, df2, x0_f2, A2))
    print("f3:", constrained_steepest_descent(f3, df3, x0_f3, A3))
    print("f4:", constrained_steepest_descent(f4, df4, x0_f4, A4))
    print("f5:", constrained_steepest_descent(f5, df5, x0_f5, A5))
    
testPrint()